In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
BASE_PATH = "/Workspace/Users/nabanya04@gmail.com"
ORDERS_PATH = f"{BASE_PATH}/orders.csv"
CUSTOMERS_PATH = f"{BASE_PATH}/customers.csv"
STATUS_PATH = f"{BASE_PATH}/delivery_status.csv"
DELTA_PATH = "/Volumes/workspace/default/my_vol/customer_orders/orders_with_latest_status"

1. Load cleaned order data

In [0]:
read_csv = lambda p: spark.read.option("header", True).option("inferSchema", True).csv(p)


In [0]:
orders = (
    read_csv(ORDERS_PATH)
    .dropDuplicates(["order_id"])
    .dropna(subset=["customer_id", "delivery_date"])
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("expected_delivery_date", F.to_date("expected_delivery_date"))
    .withColumn("delivery_date", F.to_date("delivery_date"))
    .withColumn("delay_days", F.datediff("delivery_date", "expected_delivery_date"))
    .withColumn("is_delayed", (F.col("delay_days") > 0).cast("int"))
)
customers = read_csv(CUSTOMERS_PATH)
status_history = read_csv(STATUS_PATH).withColumn("status_date", F.to_date("status_date"))

print(f"Orders: {orders.count()} | Customers: {customers.count()} | Status rows: {status_history.count()}")

Orders: 298 | Customers: 30 | Status rows: 1200


2. Update latest delivery status

In [0]:
latest_window = Window.partitionBy("order_id").orderBy(F.desc("status_date"), F.desc("status"))
latest_status = (
    status_history
    .withColumn("rn", F.row_number().over(latest_window))
    .filter("rn = 1")
    .select("order_id", F.col("status").alias("latest_status"), F.col("status_date").alias("status_updated_on"))
)


In [0]:
final_df = (
    orders.join(latest_status, on="order_id", how="left")
          .join(customers, on="customer_id", how="left")
)
display(final_df.limit(10))

customer_id,order_id,order_date,expected_delivery_date,delivery_date,delivery_issue,delay_days,is_delayed,latest_status,status_updated_on,customer_name,email,region
16,ORD0028,2026-06-01,2026-06-09,2026-06-08,null,-1,0,DELIVERED,2026-06-08,Customer 16,customer16@example.com,West
23,ORD0058,2026-07-23,2026-07-30,2026-08-01,Address not found,2,1,DELAYED,2026-08-01,Customer 23,customer23@example.com,South
7,ORD0073,2026-06-04,2026-06-10,2026-06-12,Weather,2,1,DELAYED,2026-06-12,Customer 7,customer7@example.com,West
25,ORD0119,2026-05-08,2026-05-12,2026-05-12,null,0,0,DELIVERED,2026-05-12,Customer 25,customer25@example.com,East
25,ORD0122,2026-07-22,2026-07-25,2026-07-31,Damaged package,6,1,DELAYED,2026-07-31,Customer 25,customer25@example.com,East
14,ORD0127,2026-08-19,2026-08-28,2026-08-29,Weather,1,1,DELAYED,2026-08-29,Customer 14,customer14@example.com,North
12,ORD0141,2026-08-21,2026-08-26,2026-09-01,Stock shortage,6,1,DELAYED,2026-09-01,Customer 12,customer12@example.com,West
23,ORD0155,2026-05-17,2026-05-26,2026-05-30,Address not found,4,1,DELAYED,2026-05-30,Customer 23,customer23@example.com,South
17,ORD0206,2026-02-02,2026-02-06,2026-02-07,Damaged package,1,1,DELAYED,2026-02-07,Customer 17,customer17@example.com,North
19,ORD0223,2026-02-16,2026-02-22,2026-02-26,Courier delay,4,1,DELAYED,2026-02-26,Customer 19,customer19@example.com,North


3. Save results as Delta

In [0]:
final_df.write.format("delta").mode("overwrite").save(DELTA_PATH)
final_df.write.format("delta").mode("overwrite").saveAsTable("orders_with_latest_status")

4. Top 5 delayed customers (SQL)

In [0]:
%sql
SELECT customer_id, customer_name, region, SUM(is_delayed) AS delayed_orders
FROM orders_with_latest_status GROUP BY customer_id, customer_name, region
ORDER BY delayed_orders DESC LIMIT 5

customer_id,customer_name,region,delayed_orders
23,Customer 23,South,14
14,Customer 14,North,11
6,Customer 6,West,11
15,Customer 15,South,9
25,Customer 25,East,9
